# Labwork 2 — Gradient descent and line search

**Week 2 · Day 2 · ≈ 170 min at the keyboard**

Read Lecture 2 first. This labwork fills in part of the `optlab` package you cloned;
you edit the real source files, and the notebook checks your work as you go.

**What you build today:** the one descent loop, written once and reused by every deterministic optimizer this week

**Files you will open:**

- `linesearch.py`, `stopping.py`, `observers.py`
- `optimizers/descent.py`, `optimizers/directions.py`

> **The rule.** `src/optlab/interfaces.py`, `results.py`, `errors.py` and `types.py` are
> **provided** — never edit them. Everything else under `src/optlab/` is yours: replace
> each `raise NotImplementedError` with working code, keeping the signature and honouring
> the docstring.

## Setup

Run this once. It points the notebook at your `optlab` clone and gives you a
`check()` helper that runs a specific test file and reports what happened.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Adjust if your clone lives elsewhere.
OPTLAB = (Path.cwd() / ".." / ".." / "optlab").resolve()
assert OPTLAB.exists(), f"optlab not found at {OPTLAB} -- edit OPTLAB above"
print("optlab:", OPTLAB)


def check(*pytest_args):
    """Run pytest inside the optlab clone and show a short report."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", *pytest_args],
                       cwd=OPTLAB, capture_output=True, text=True)
    out = r.stdout + r.stderr
    tail = [l for l in out.splitlines() if l.strip()][-12:]
    print("\n".join(tail))
    if "No module named pytest" in out:
        verdict = "pytest is not installed -- run:  pip install -e '.[dev]'"
    elif r.returncode == 0:
        verdict = "PASSED"
    elif r.returncode == 5:
        # Exit code 5 means pytest collected nothing at all. That is NOT a failure of
        # your code: no test in the suite matches what was asked for. Some days have no
        # automated tests yet; judge those exercises by the checks written in the text.
        verdict = "no tests matched -- nothing to run here, this is not a failure"
    else:
        verdict = "not yet -- keep going"
    print()
    print(verdict)


def edit(relpath):
    """Print the absolute path of a source file, so you can open it in the editor."""
    print(OPTLAB / "src" / "optlab" / relpath)


check("tests/test_no_oracle_in_src.py")   # provided, and already green

---

## Exercise 1 — The one loop  *(≈ 55 min)*

Implement, in this order: `GradientNormBelow`, `MaxIterations`, `AnyOf`
(`stopping.py`); `History` (`observers.py`); `FixedStep` (`linesearch.py`);
`SteepestDescent` (`directions.py`); then `DescentOptimizer.minimize` (`descent.py`).

The loop does six things and nothing else: evaluate, ask for a direction, ask for a step,
move, emit a `StepEvent` to the observers, ask the criterion whether to stop.

Two rules it must respect:

- `converged=True` **only** when a genuine convergence test fired. Hitting `MaxIterations`
  gives `converged=False`, with the reason in `message`.
- one `StepEvent` per iteration, no more.

**You will not edit this file again this week.** Newton on day 4 is a new `DirectionRule`;
Armijo tomorrow is a new `LineSearch`. If either forces you back in here, something has
been put in the wrong class.

**Open:** `src/optlab/stopping.py`, `observers.py`, `linesearch.py`, `optimizers/directions.py`, `optimizers/descent.py`

In [ ]:
edit("optimizers/descent.py")
check("-m", "day2")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

`minimize` takes `objective: object` because the family is wider than `Objective` — narrow it locally with an assignment and let mypy infer. Build the `StepEvent` *after* moving, so `x` and `value` describe where you landed.

</details>

---

## Exercise 2 — Armijo backtracking  *(≈ 55 min)*

Implement `Armijo.step`.

Start at `alpha0`, and while `f(x + αd) > f(x) + c₁·α·gᵀd`, multiply `α` by `rho`. After
`max_backtracks` failures, raise `LineSearchFailed` — and `DescentOptimizer` must **catch**
it and return `converged=False`, not crash. Go back and add that if you did not.

Verify the hand example from the lecture: `f(x) = x²` at `x = 1` rejects `α = 1` and
accepts `α = 0.5`.

Then the point of the exercise: build a `DescentOptimizer` with `Armijo` instead of
`FixedStep` and run it. **Confirm you did not touch `descent.py`.**

**Open:** `src/optlab/linesearch.py`

In [ ]:
check("tests/contracts", "-m", "day2")

import subprocess
d = subprocess.run(["git", "diff", "--stat", "src/optlab/optimizers/descent.py"],
                   cwd=OPTLAB, capture_output=True, text=True).stdout.strip()
print("\nchanges to descent.py since Exercise 1:", d if d else "none -- open/closed held")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

Guard `slope = g @ d` and raise `ValueError` if it is non-negative: a non-descent direction is a bug in the caller, not something to backtrack around.

</details>

---

## Exercise 3 — Momentum, rates, and a deliberate explosion  *(≈ 50 min)*

Implement `HeavyBall`. It is **stateful** — it stores the previous direction — which
is why a `DirectionRule` is an object rather than a function.

Then three measurements on `Quadratic.ill_conditioned(n=2, kappa=100)`:

1. **Recover the rate.** Run steepest descent with `α = 2/(L+μ)` and check the observed
   contraction against the predicted `(κ−1)/(κ+1)`. They should agree to a few decimals.
2. **Count the steps.** Steepest descent to relative error `1e-6`, then heavy ball. You
   should see roughly `690` against roughly `90`. Note the measured heavy-ball count
   exceeds the asymptotic prediction of `69` — the bound describes the tail, and the
   iteration spirals before it settles.
3. **Break it.** Set `FixedStep` above `2/L` and watch the iterate diverge. Then explain
   it with the descent lemma.

Step 3 is not optional. An algorithm you have only seen succeed is one you do not yet
understand.

**Open:** `src/optlab/optimizers/directions.py`

In [ ]:
check("-m", "day2")

---

## Checkpoint

Everything from day 1 to day 2 should be green before you leave, and
`mypy --strict` must be clean. A red type check counts as a failure.

In [ ]:
check("-m", "day1 or day2")

In [ ]:
r = subprocess.run([sys.executable, "-m", "mypy"], cwd=OPTLAB,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

---

## Before the debrief

Be able to answer:

1. Why is `α ≤ 1/L` safe — and where exactly does the `2` in `2/L` come from?
2. Why does momentum help? Answer in terms of what happens to the oscillating components
   versus the drift along the valley floor.
3. Which of your classes would have to change to add a new direction rule? (Correct
   answer: none.)